# Survey: Software Engineering in the era of GenAI

In [2]:
# Step 1: Importing and Reading

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FILE_PATH = "./data/Survey_SoftwareDevelopmentInEraGenAI_polished.xlsx"
SHEETS = {}

def load_data():
    global SHEETS
    SHEETS = pd.read_excel(FILE_PATH, sheet_name=None, header=None)
    print(f"Loaded {len(SHEETS)} sheets: {list(SHEETS.keys())}")

load_data()

Loaded 24 sheets: ['complete', 'Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23']


In [3]:
# Step 2: Parsing data

def get_standard_question(qid):
    """For Q01-Q09, Q11-Q13, Q16-Q23: returns DataFrame with columns [id, response]"""
    df = SHEETS[qid].iloc[1:].copy()
    df.columns = ['id', 'response']
    return df.reset_index(drop=True)

def get_scaling_question(qid):
    """For Q10, Q14: returns DataFrame with id as index, options as columns"""
    raw = SHEETS[qid]
    options = raw.iloc[1, 1:].tolist()
    df = raw.iloc[2:].copy()
    df.columns = ['id'] + options
    return df.set_index('id')

def get_multichoice(qid):
    """For semicolon-separated questions (e.g. Q11, Q16): returns Series of lists"""
    df = get_standard_question(qid)
    df['choices'] = df['response'].apply(
        lambda x: [c.strip() for c in str(x).split(';') if c.strip()] if pd.notna(x) else []
    )
    return df

In [4]:
# Step 3: Validation of data extraction
def validate():
    print("Q10 id=3:", get_scaling_question('Q10').loc[3].dropna().to_dict())
    print("Q10 id=24:", get_scaling_question('Q10').loc[24].dropna().to_dict())
    q08 = get_standard_question('Q08')
    print("Q08 id=2:", q08[q08['id']==2]['response'].values[0])
    q23 = get_standard_question('Q23')
    print("Q23 id=6:", q23[q23['id']==6]['response'].values[0])
    q16 = get_multichoice('Q16')
    print("Q16 id=14:", q16[q16['id']==14]['choices'].values[0])

validate()

Q10 id=3: {'Brainstorming': 'Somewhat Useful', 'Self-development and Learning': 'Very Useful', 'Code generation / auto-completion': 'Very Useful', 'Refactoring existing code': 'Very Useful', 'Test / Test case generation': 'Very Useful', 'Debugging / Error Explanation': 'Not Applicable', 'Code Documentation': 'Very Useful', 'Code Translation / Migration': 'Not Applicable', 'Code Review': 'Not Applicable', 'CI/CD Automation': 'Not Applicable', 'Maintenance Automation': 'Not Applicable', 'Security analysis': 'Not Applicable'}
Q10 id=24: {'Brainstorming': 'Very Useful', 'Code generation / auto-completion': 'Essential'}
Q08 id=2: Regularly (daily or almost daily)
Q23 id=6: Low-Code is dead; Development by Context Engineering is the new thing. Get used to it!
Q16 id=14: ['Critical thinking and code review skills', 'Secure coding and ethical AI awareness', 'Data privacy and regulatory compliance awareness']


In [5]:
# Step 4: Generating general code for visualizing single-choice categorical questions

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# ── Palette ────────────────────────────────────────────────────────────────────
PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
    "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD",
]

def plot_pie(
    sheets: dict,
    qid: str,
    *,
    order: list = None,          # optional: enforce slice order
    colors: list = None,         # optional: override palette
    min_pct_label: float = 3.0,  # hide label if slice < this %
    figsize: tuple = (8, 5),
    save_path: str = None,
):
    """
    Render a pie chart for a single-choice categorical survey question.

    Parameters
    ----------
    sheets       : dict returned by pd.read_excel(..., sheet_name=None, header=None)
    qid          : sheet name, e.g. 'Q08'
    order        : list of answer strings to fix slice order (subset allowed)
    colors       : list of hex/named colours; falls back to PALETTE
    min_pct_label: slices smaller than this percentage won't show a label
    figsize      : (width, height) in inches
    save_path    : if given, save figure to this path instead of showing
    """
    sheet    = sheets[qid]
    question = sheet.iloc[0, 1]                        # question text
    responses = sheet.iloc[1:, 1].dropna().astype(str) # answers

    counts = Counter(responses)

    # apply ordering if requested
    if order:
        labels = [o for o in order if o in counts] + \
                 [k for k in counts if k not in order]
    else:
        labels = sorted(counts, key=lambda k: -counts[k])

    sizes  = [counts[l] for l in labels]
    total  = sum(sizes)
    clrs   = (colors or PALETTE)[:len(labels)]

    # ── figure ─────────────────────────────────────────────────────────────────
    fig, (ax_pie, ax_legend) = plt.subplots(
        1, 2, figsize=figsize,
        gridspec_kw={"width_ratios": [1, 1]},
    )
    fig.patch.set_facecolor("#F7F9FC")

    # autopct: hide tiny labels
    def autopct(pct):
        return f"{pct:.1f}%" if pct >= min_pct_label else ""

    wedges, texts, autotexts = ax_pie.pie(
        sizes,
        labels=None,
        colors=clrs,
        autopct=autopct,
        startangle=140,
        pctdistance=0.75,
        wedgeprops={"linewidth": 1.2, "edgecolor": "white"},
    )
    for at in autotexts:
        at.set_fontsize(9)
        at.set_color("white")
        at.set_fontweight("bold")

    ax_pie.set_facecolor("#F7F9FC")

    # ── legend (right panel) ───────────────────────────────────────────────────
    ax_legend.set_facecolor("#F7F9FC")
    ax_legend.axis("off")

    legend_handles = []
    for lbl, sz, clr in zip(labels, sizes, clrs):
        pct = sz / total * 100
        # wrap long labels at 40 chars
        display = "\n".join([lbl[i:i+40] for i in range(0, len(lbl), 40)])
        patch = mpatches.Patch(color=clr,
                               label=f"{display}\n  n={sz}  ({pct:.1f}%)")
        legend_handles.append(patch)

    ax_legend.legend(
        handles=legend_handles,
        loc="center left",
        frameon=False,
        fontsize=8.5,
        labelspacing=1.0,
        handleheight=1.4,
        handlelength=1.2,
    )

    # ── title ──────────────────────────────────────────────────────────────────
    # wrap question text at 70 chars
    import textwrap
    wrapped = "\n".join(textwrap.wrap(question, width=70))
    fig.suptitle(
        f"{qid}  |  {wrapped}",
        fontsize=10,
        fontweight="bold",
        color="#2C3E50",
        x=0.5, y=1.02,
        ha="center",
        wrap=True,
    )
    fig.text(0.5, -0.01, f"Total responses: {total}", ha="center",
             fontsize=8, color="#7F8C8D")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        print(f"Saved → {save_path}")
    else:
        plt.show()
    plt.close()

In [6]:
import textwrap
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── shared palette (reuse from Step 3) ────────────────────────────────────────
PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
    "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD",
]

def plot_multichoice(
    sheets: dict,
    qid: str,
    *,
    order: list = None,           # enforce bar order (top → bottom)
    top_n: int = None,            # show only top-N choices
    wrap_width: int = 42,         # chars before wrapping y-axis labels
    show_pct: bool = True,        # show % of respondents on bars
    color: str = None,            # single bar colour; falls back to PALETTE[0]
    figsize: tuple = None,        # auto-sized if None
    save_path: str = None,
):
    """
    Horizontal bar chart for multiple-choice (select-all / select-up-to-N)
    survey questions.

    Parameters
    ----------
    sheets      : dict from pd.read_excel(..., sheet_name=None, header=None)
    qid         : sheet key, e.g. 'Q11'
    order       : explicit label order (top to bottom); unmatched labels appended
    top_n       : keep only the N most frequent choices
    wrap_width  : max chars per line for y-axis labels
    show_pct    : annotate bars with "n  (x%)" where % = share of respondents
    color       : hex/named colour for all bars
    figsize     : (width, height); auto-calculated from number of choices if None
    save_path   : file path to save; shows interactively if None
    """
    sheet     = sheets[qid]
    question  = sheet.iloc[0, 1]
    raw       = sheet.iloc[1:, 1].dropna().astype(str)
    n_resp    = len(raw)           # unique respondents who answered

    # ── parse semicolon-separated choices ─────────────────────────────────────
    all_choices = []
    for cell in raw:
        all_choices.extend([c.strip() for c in cell.split(';') if c.strip()])
    counts = Counter(all_choices)

    # ── ordering ───────────────────────────────────────────────────────────────
    if order:
        labels = [o for o in order if o in counts] + \
                 [k for k in sorted(counts, key=lambda k: -counts[k])
                  if k not in order]
    else:
        labels = sorted(counts, key=lambda k: counts[k])   # ascending → bars read top-down as most → least

    if top_n:
        top_labels = {k for k, _ in counts.most_common(top_n)}
        labels = [l for l in labels if l in top_labels]

    values = [counts[l] for l in labels]
    pcts   = [v / n_resp * 100 for v in values]

    # ── wrap long labels ───────────────────────────────────────────────────────
    wrapped_labels = [
        "\n".join(textwrap.wrap(l, width=wrap_width)) for l in labels
    ]

    # ── figure sizing ──────────────────────────────────────────────────────────
    n_bars = len(labels)
    if figsize is None:
        figsize = (10, max(3.5, 0.55 * n_bars + 1.5))

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    bar_color = color or PALETTE[0]
    bars = ax.barh(
        wrapped_labels, values,
        color=bar_color, edgecolor="white", linewidth=0.8,
        height=0.6,
    )

    # ── subtle grid ───────────────────────────────────────────────────────────
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(axis="x", color="#D5DCE8", linewidth=0.7, linestyle="--", zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#C0C8D4")
    ax.tick_params(axis="y", length=0, labelsize=8.5)
    ax.tick_params(axis="x", labelsize=8)
    ax.set_xlabel("Number of respondents", fontsize=8.5, color="#5A6475")

    # ── bar annotations ────────────────────────────────────────────────────────
    x_max = max(values) if values else 1
    for bar, v, pct in zip(bars, values, pcts):
        label_str = f"  {v}  ({pct:.0f}%)" if show_pct else f"  {v}"
        ax.text(
            bar.get_width() + x_max * 0.01,
            bar.get_y() + bar.get_height() / 2,
            label_str,
            va="center", ha="left",
            fontsize=8, color="#2C3E50", fontweight="bold",
        )
    ax.set_xlim(0, x_max * 1.22)   # room for annotation

    # ── title & footnote ───────────────────────────────────────────────────────
    wrapped_q = "\n".join(textwrap.wrap(question, width=80))
    ax.set_title(
        f"{qid}  |  {wrapped_q}",
        fontsize=9.5, fontweight="bold", color="#2C3E50",
        loc="left", pad=10,
    )
    fig.text(
        0.0, -0.03,
        f"n = {n_resp} respondents  •  % = share of respondents  •  multiple selections allowed",
        fontsize=7.5, color="#7F8C8D",
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        print(f"Saved → {save_path}")
    else:
        plt.show()
    plt.close()

In [7]:
import textwrap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# ── Default scale definitions (reuse / extend as needed) ──────────────────────

SCALE_USEFULNESS = {
    "order":  ["Not Useful", "Somewhat Useful", "Very Useful", "Essential"],
    "na":     "Not Applicable",         # treated separately, not stacked
    "colors": ["#C44E52", "#DD8452", "#4C72B0", "#2A6099"],
}

SCALE_IMPORTANCE_CHANGE = {
    "order":  ["Decreasing in Importance", "No Change",
               "Slightly Increasing", "Strongly Increasing"],
    "na":     None,
    "colors": ["#C44E52", "#8C8C8C", "#55A868", "#2A6099"],
}


def plot_scaling(
    sheets: dict,
    qid: str,
    scale: dict,
    *,
    item_order: list = None,     # fix row order; unmatched appended at bottom
    top_n: int = None,           # show only top-N items (sorted by positive %)
    wrap_width: int = 38,        # chars before wrapping item labels
    show_na_bar: bool = True,    # append a thin "N/A" segment at right edge
    show_counts: bool = True,    # annotate segments with n= when wide enough
    min_seg_pct: float = 6.0,    # hide count label if segment < this %
    figsize: tuple = None,       # auto-sized if None
    save_path: str = None,
):
    """
    Stacked horizontal bar chart (100 %) for Likert / rating-scale matrix questions.

    Parameters
    ----------
    sheets       : dict from pd.read_excel(..., sheet_name=None, header=None)
    qid          : sheet key, e.g. 'Q10' or 'Q14'
    scale        : dict with keys:
                     'order'  – list of scale levels low → high
                     'colors' – matching list of hex colours
                     'na'     – string label for N/A level, or None
    item_order   : explicit row order (top → bottom); unmatched appended
    top_n        : keep only the N rows with highest positive-end %
    wrap_width   : y-axis label wrap width in characters
    show_na_bar  : draw a thin grey N/A block beyond the 100 % bar
    show_counts  : annotate each segment with respondent count
    min_seg_pct  : suppress annotation when segment share < this %
    figsize      : (width, height); auto from row count if None
    save_path    : file path to save; shows interactively if None
    """
    # ── load data ──────────────────────────────────────────────────────────────
    sheet    = sheets[qid]
    question = sheet.iloc[0, 1]
    items    = sheet.iloc[1, 1:].tolist()
    raw      = sheet.iloc[2:].copy()
    raw.columns = ['id'] + items
    raw = raw.set_index('id')

    scale_order = scale["order"]
    na_label    = scale.get("na")
    colors      = scale["colors"]

    # ── build count matrix  (items × scale levels) ────────────────────────────
    records = {}
    na_counts = {}
    for item in items:
        col     = raw[item].dropna().astype(str)
        na_n    = (col == na_label).sum() if na_label else 0
        valid   = col[col != na_label] if na_label else col
        n_valid = len(valid)
        counts  = {lvl: (valid == lvl).sum() for lvl in scale_order}
        records[item]   = {"counts": counts, "n_valid": n_valid}
        na_counts[item] = na_n

    # ── item ordering ──────────────────────────────────────────────────────────
    if item_order:
        ordered = [i for i in item_order if i in records] + \
                  [i for i in items if i not in item_order]
    else:
        # default: sort by share of top-2 positive levels descending
        def positive_pct(item):
            d = records[item]
            pos = sum(d["counts"].get(lvl, 0) for lvl in scale_order[-2:])
            return pos / d["n_valid"] if d["n_valid"] else 0
        ordered = sorted(items, key=positive_pct)   # ascending → top of chart = most positive

    if top_n:
        def pos_pct(i):
            d = records[i]
            pos = sum(d["counts"].get(lvl, 0) for lvl in scale_order[-2:])
            return pos / d["n_valid"] if d["n_valid"] else 0
        ordered = sorted(ordered, key=pos_pct)[-top_n:]

    n_rows = len(ordered)

    # ── figure ─────────────────────────────────────────────────────────────────
    if figsize is None:
        figsize = (12, max(4.0, 0.52 * n_rows + 2.2))

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    bar_h    = 0.62
    na_width = 0.04   # fraction of axis width reserved for N/A tick marks

    y_positions = np.arange(n_rows)

    # ── stacked bars ───────────────────────────────────────────────────────────
    for yi, item in enumerate(ordered):
        d       = records[item]
        n_valid = d["n_valid"] if d["n_valid"] > 0 else 1
        x_left  = 0.0

        for lvl, clr in zip(scale_order, colors):
            n  = d["counts"].get(lvl, 0)
            pct = n / n_valid * 100
            ax.barh(yi, pct, left=x_left, height=bar_h,
                    color=clr, edgecolor="white", linewidth=0.6)

            # segment annotation
            if show_counts and pct >= min_seg_pct:
                ax.text(x_left + pct / 2, yi, str(n),
                        ha="center", va="center",
                        fontsize=7.5, color="white", fontweight="bold")
            x_left += pct

        # N/A tick at right margin
        if show_na_bar and na_label and na_counts[item] > 0:
            ax.text(102, yi, f"N/A={na_counts[item]}",
                    va="center", ha="left",
                    fontsize=7, color="#7F8C8D")

    # ── axes cosmetics ─────────────────────────────────────────────────────────
    ax.set_xlim(0, 100)
    ax.set_ylim(-0.6, n_rows - 0.4)
    ax.set_xlabel("% of valid respondents", fontsize=8.5, color="#5A6475")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.grid(axis="x", color="#D5DCE8", linewidth=0.6, linestyle="--", zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#C0C8D4")
    ax.tick_params(axis="y", length=0, labelsize=8)
    ax.tick_params(axis="x", labelsize=8)

    wrapped = ["\n".join(textwrap.wrap(i, width=wrap_width)) for i in ordered]
    ax.set_yticks(y_positions)
    ax.set_yticklabels(wrapped)

    # ── legend ─────────────────────────────────────────────────────────────────
    handles = [mpatches.Patch(color=c, label=l)
               for c, l in zip(colors, scale_order)]
    ax.legend(
        handles=handles,
        loc="lower right",
        bbox_to_anchor=(1.0, 1.01),
        ncol=len(scale_order),
        frameon=False,
        fontsize=8,
    )

    # ── title & footnote ───────────────────────────────────────────────────────
    wrapped_q = "\n".join(textwrap.wrap(question, width=90))
    ax.set_title(
        f"{qid}  |  {wrapped_q}",
        fontsize=9.5, fontweight="bold", color="#2C3E50",
        loc="left", pad=28,
    )
    n_total = raw.shape[0]
    note = f"n = {n_total} respondents"
    if na_label:
        note += "  •  % excludes 'Not Applicable' responses"
    fig.text(0.0, -0.02, note, fontsize=7.5, color="#7F8C8D")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        print(f"Saved → {save_path}")
    else:
        plt.show()
    plt.close()

In [8]:
import textwrap
import numpy as np
import pandas as pd
from collections import Counter
from scipy.stats import chisquare, entropy as scipy_entropy

# ══════════════════════════════════════════════════════════════════════════════
#  INTERNAL HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _parse_responses(sheets, qid, is_multi=False):
    """Return (question_str, Series of raw responses, n_respondents)."""
    sheet     = sheets[qid]
    question  = str(sheet.iloc[0, 1])
    raw       = sheet.iloc[1:, 1].dropna().astype(str)
    n_resp    = len(raw)
    if is_multi:
        choices = []
        for cell in raw:
            choices.extend([c.strip() for c in cell.split(';') if c.strip()])
        return question, pd.Series(choices), n_resp
    return question, raw, n_resp


def _parse_scaling(sheets, qid, scale_order, na_label=None):
    """Return (question_str, DataFrame items×levels with counts, n_respondents)."""
    sheet    = sheets[qid]
    question = str(sheet.iloc[0, 1])
    items    = sheet.iloc[1, 1:].tolist()
    raw      = sheet.iloc[2:].copy()
    raw.columns = ['id'] + items
    raw = raw.set_index('id')
    n_resp = raw.shape[0]
    return question, raw, items, n_resp


def _wilson_ci(n_success, n_total, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n_total == 0:
        return (0.0, 0.0)
    p = n_success / n_total
    denom = 1 + z**2 / n_total
    centre = p + z**2 / (2 * n_total)
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n_total)) / n_total)
    return (round((centre - margin) / denom, 4),
            round((centre + margin) / denom, 4))


def _shannon_entropy(counts_array):
    """Shannon entropy in bits and normalised [0,1]."""
    total = sum(counts_array)
    if total == 0:
        return 0.0, 0.0
    probs  = np.array(counts_array) / total
    probs  = probs[probs > 0]
    h      = float(-np.sum(probs * np.log2(probs)))
    h_max  = np.log2(len(counts_array))
    h_norm = round(h / h_max, 4) if h_max > 0 else 0.0
    return round(h, 4), h_norm


def _chi2_uniform(counts_array):
    """Chi-squared goodness-of-fit against a uniform distribution."""
    from scipy.stats import chisquare
    if len(counts_array) < 2:
        return None, None
    chi2, p = chisquare(counts_array)
    return round(float(chi2), 4), round(float(p), 6)


def _ordinal_stats(values_series, scale_map):
    """
    Map text labels → numeric scores, then return descriptive + inferential stats.
    Includes 95 % bootstrap CI on the mean (n_boot=2000).
    """
    numeric = values_series.map(scale_map).dropna()
    if numeric.empty:
        return {}
    n      = len(numeric)
    mean_  = round(float(numeric.mean()), 3)
    std_   = round(float(numeric.std(ddof=1)), 3)
    med_   = round(float(numeric.median()), 3)
    # bootstrap CI on mean
    rng    = np.random.default_rng(42)
    boots  = [rng.choice(numeric.values, size=n, replace=True).mean()
              for _ in range(2000)]
    ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])
    return {
        "n": n,
        "mean": mean_,
        "std": std_,
        "median": med_,
        "mean_ci_95": (round(ci_lo, 3), round(ci_hi, 3)),
    }


# ══════════════════════════════════════════════════════════════════════════════
#  PUBLIC API
# ══════════════════════════════════════════════════════════════════════════════

def analyse_question(
    sheets: dict,
    qid: str,
    question_type: str,             # 'single' | 'multi' | 'scaling'
    *,
    # ── categorical options ────────────────────────────────────────────────
    top_n: int = None,              # report only top-N categories
    # ── scaling options ───────────────────────────────────────────────────
    scale: dict = None,             # same scale dict used in plot_scaling()
    na_label: str = None,           # label to exclude from numeric scoring
    # ── output control ────────────────────────────────────────────────────
    print_report: bool = True,      # pretty-print to notebook
    return_results: bool = False,   # also return raw dict
) -> dict | None:
    """
    Compute statistical summary for any survey question.

    Parameters
    ----------
    sheets         : dict from pd.read_excel(..., sheet_name=None, header=None)
    qid            : sheet key, e.g. 'Q08', 'Q11', 'Q10'
    question_type  : 'single'  – one answer per respondent
                     'multi'   – semicolon-separated answers
                     'scaling' – rating matrix (Q10 / Q14 style)
    top_n          : for categorical types, report only top-N categories
    scale          : for 'scaling' type, same dict used in plot_scaling()
                     must contain keys: 'order', 'colors', 'na' (optional)
    na_label       : overrides scale['na'] if given directly
    print_report   : print formatted report to stdout / notebook cell
    return_results : return the raw results dict (useful for downstream analysis)

    Returns
    -------
    dict (or None if return_results=False) with keys depending on type:

    'single' / 'multi'
        question, n_respondents, n_answers, counts, proportions,
        wilson_ci, dominant_category, entropy_bits, entropy_norm,
        chi2_uniform, p_uniform

    'scaling'
        question, n_respondents, per_item stats:
          counts, proportions, ordinal_stats (mean/std/median/CI),
          entropy_bits, entropy_norm, dominant_level, positive_pct
    """
    results = {"qid": qid, "question_type": question_type}

    # ══════════════════════════════════════════════════════════════════════
    #  CATEGORICAL  (single & multi)
    # ══════════════════════════════════════════════════════════════════════
    if question_type in ("single", "multi"):
        is_multi = question_type == "multi"
        question, choices, n_resp = _parse_responses(sheets, qid, is_multi)
        results["question"]      = question
        results["n_respondents"] = n_resp
        results["n_answers"]     = len(choices)   # > n_resp for multi

        counts = Counter(choices)
        if top_n:
            counts = dict(counts.most_common(top_n))
        else:
            counts = dict(counts.most_common())

        total   = sum(counts.values())
        props   = {k: round(v / n_resp, 4) for k, v in counts.items()}
        ci_dict = {k: _wilson_ci(v, n_resp) for k, v in counts.items()}
        dominant = max(counts, key=counts.get)

        h_bits, h_norm = _shannon_entropy(list(counts.values()))
        chi2, p_val    = _chi2_uniform(list(counts.values()))

        results.update({
            "counts":            counts,
            "proportions":       props,
            "wilson_ci_95":      ci_dict,
            "dominant_category": dominant,
            "entropy_bits":      h_bits,
            "entropy_norm":      h_norm,
            "chi2_uniform":      chi2,
            "p_uniform":         p_val,
        })

        if print_report:
            _print_categorical(results)

    # ══════════════════════════════════════════════════════════════════════
    #  SCALING
    # ══════════════════════════════════════════════════════════════════════
    elif question_type == "scaling":
        if scale is None:
            raise ValueError("Provide a `scale` dict for question_type='scaling'.")

        scale_order = scale["order"]
        na          = na_label or scale.get("na")
        question, raw, items, n_resp = _parse_scaling(sheets, qid, scale_order, na)
        scale_map   = {lvl: i + 1 for i, lvl in enumerate(scale_order)}

        results["question"]      = question
        results["n_respondents"] = n_resp
        results["scale_order"]   = scale_order
        results["items"]         = {}

        for item in items:
            col     = raw[item].dropna().astype(str)
            na_n    = int((col == na).sum()) if na else 0
            valid   = col[col != na] if na else col
            n_valid = len(valid)

            counts  = {lvl: int((valid == lvl).sum()) for lvl in scale_order}
            props   = {lvl: round(counts[lvl] / n_valid, 4) if n_valid else 0.0
                       for lvl in scale_order}
            dominant = max(counts, key=counts.get)
            pos_pct  = round(sum(counts.get(l, 0)
                                 for l in scale_order[-2:]) / n_valid * 100, 1) \
                       if n_valid else 0.0

            h_bits, h_norm = _shannon_entropy(list(counts.values()))
            ord_stats      = _ordinal_stats(valid, scale_map)

            results["items"][item] = {
                "n_valid":       n_valid,
                "n_na":          na_n,
                "counts":        counts,
                "proportions":   props,
                "dominant_level": dominant,
                "positive_pct":  pos_pct,
                "ordinal_stats": ord_stats,
                "entropy_bits":  h_bits,
                "entropy_norm":  h_norm,
            }

        if print_report:
            _print_scaling(results)

    else:
        raise ValueError(f"Unknown question_type='{question_type}'. "
                         "Use 'single', 'multi', or 'scaling'.")

    return results if return_results else None


# ══════════════════════════════════════════════════════════════════════════════
#  PRETTY PRINTERS
# ══════════════════════════════════════════════════════════════════════════════

def _print_categorical(r):
    sep  = "─" * 72
    sep2 = "─" * 72
    q_wrapped = "\n  ".join(textwrap.wrap(r["question"], 68))
    print(f"\n{'═'*72}")
    print(f"  {r['qid']}  [{r['question_type']}]")
    print(f"  {q_wrapped}")
    print(f"{'═'*72}")
    print(f"  Respondents : {r['n_respondents']}"
          + (f"   |   Total selections : {r['n_answers']}"
             if r['n_answers'] != r['n_respondents'] else ""))
    print(sep)
    print(f"  {'Category':<46} {'n':>4}  {'%':>6}  {'95% CI':>16}")
    print(sep)
    for cat, cnt in r["counts"].items():
        pct   = r["proportions"][cat] * 100
        lo, hi = r["wilson_ci_95"][cat]
        label = cat if len(cat) <= 46 else cat[:43] + "..."
        print(f"  {label:<46} {cnt:>4}  {pct:>5.1f}%  [{lo:.3f}, {hi:.3f}]")
    print(sep)
    print(f"  Dominant    : {r['dominant_category']}")
    print(f"  Entropy     : {r['entropy_bits']:.3f} bits  "
          f"(normalised: {r['entropy_norm']:.3f}  — "
          f"0=consensus, 1=uniform)")
    if r["chi2_uniform"] is not None:
        sig = "✓ significant" if r["p_uniform"] < 0.05 else "✗ not significant"
        print(f"  χ² (uniform): {r['chi2_uniform']:.3f}   p = {r['p_uniform']:.4f}  "
              f"→ {sig} at α=0.05")
    print()


def _print_scaling(r):
    sep = "─" * 88
    q_wrapped = "\n  ".join(textwrap.wrap(r["question"], 84))
    print(f"\n{'═'*88}")
    print(f"  {r['qid']}  [scaling]")
    print(f"  {q_wrapped}")
    print(f"  Scale : {' → '.join(r['scale_order'])}")
    print(f"{'═'*88}")
    print(f"  n = {r['n_respondents']} respondents\n")

    col_w = 28
    for item, d in r["items"].items():
        label = (item[:col_w - 2] + "..") if len(item) > col_w else item
        os    = d["ordinal_stats"]
        print(f"  ┌─ {label}")
        dist  = "  ".join(
            f"{lvl[:14]}: {d['counts'][lvl]:>2} ({d['proportions'][lvl]*100:.0f}%)"
            for lvl in r["scale_order"]
        )
        print(f"  │  {dist}")
        if os:
            print(f"  │  mean={os['mean']} (95% CI {os['mean_ci_95']})  "
                  f"median={os['median']}  std={os['std']}")
        print(f"  │  dominant='{d['dominant_level']}'  "
              f"positive%={d['positive_pct']}%  "
              f"entropy={d['entropy_bits']:.3f} bits  "
              f"N/A={d['n_na']}")
        print()